# Imports General

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OrdinalEncoder, MinMaxScaler, OneHotEncoder

# Carga de Datos

In [ ]:
archivo1 = "clientes_banca_raw.csv"
df = pd.read_csv(archivo1)
df = df.drop(['id_cliente'], axis=1)
df = df.drop(['salario_mensual_aprox'], axis=1)
df = df.drop(['comentarios_cliente'], axis=1)
df = df.drop(['motivo_cancelacion'], axis=1)

# Partición Train / Test

In [ ]:
X = df.drop('abandono', axis=1)
Y = df['abandono']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Tabla Resumen Auditoria Inicial

| Variable | Tipo de característica | Poder Predictivo Observado | Decisión Técnica |
| :--- | :--- | :--- | :--- |
| **id_cliente** | Identificador | Ninguna porque solo es un código administrativo | Quitar la columna porque solo identifica al cliente y no sirve para aprender. |
| **edad** | Numérica continua | Casi nula ya que fieles y abandonos tienen edades muy similares | Borrar la edad imposible de 999 años, rellenar los huecos vacíos con la mediana y guardarla por si sirve por grupos de edad. |
| **ingresos_anuales** | Numérica continua | Muy débil de forma directa | Rellenar los huecos vacíos con la mediana, revisar los que tienen 0 € y retocar los sueldos más altos. |
| **salario_mensual_aprox** | Numérica continua | Totalmente repetida con los ingresos anuales | Quitar la columna porque dice exactamente lo mismo que los ingresos anuales. |
| **antiguedad_meses** | Numérica discreta | Los clientes más antiguos tienden a quedarse algo más | Rellenar las 5 filas vacías con la mediana de la antigüedad para no perder clientes. |
| **saldo_cuenta** | Numérica continua | Clave porque los clientes con más saldo casi siempre se quedan | No hay que rellenar nada y solo ajustar los saldos más ricos para no descompensar. |
| **nivel_estudios** | Categórica ordinal | Apenas influye en la decisión de marcharse | Rellenar los huecos vacíos con Grado y pasar los estudios a números ordenados de menor a mayor. |
| **tipo_cuenta** | Categórica nominal | Influye bastante ya que Ahorro pierde un 54% de clientes mientras que Inversión solo pierde un 32% | Arreglar faltas y espacios, cambiar el signo de interrogación por Corriente y convertir cada cuenta en columnas separadas. |
| **nivel_satisfaccion** | Categórica ordinal | Muy decisiva ya que el 92% de los muy descontentos cancela la cuenta | Guardar con máxima prioridad y pasar las respuestas a números del 1 al 5. |
| **num_productos** | Numérica discreta | Influye claramente porque cuantos más productos tienen contratados menos se van | No hay que tocar nada porque los números del 1 al 4 están correctos. |
| **tiene_tarjeta** | Categórica binaria | No influye porque la proporción de tarjetas es idéntica en ambos grupos | No hay que tocar nada porque ya viene lista con ceros y unos. |
| **canal_preferido** | Categórica nominal | Muestra pequeñas diferencias según la vía de contacto elegida | No hay que rellenar nada y poner cada canal en una columna distinta. |
| **comentarios_cliente** | Categórica nominal | No se puede usar porque casi todas las filas están vacías | Quitar la columna del modelo. |
| **motivo_cancelacion** | Categórica nominal | Es trampa porque solo lo rellenan los que ya se han ido | Quitar la columna para no falsear los resultados. |
| **abandono** | Categórica binaria | Es la variable objetivo y representa lo que queremos predecir | No hay que tocar nada porque está completa, equilibrada y lista con ceros y unos. |

# Preprocesamiento

## Sanitización de centinelas

In [ ]:
# Sanitización de variables numéricas y categóricas
# Sustituimos centinelas numéricos por np.nan en Train y Test
cols_con_centinelas = ['edad', 'ingresos_anuales', 'saldo_cuenta']

X_train.loc[(X_train['edad'] <= 0) | (X_train['edad'] > 120), 'edad'] = np.nan
X_train.loc[X_train['ingresos_anuales'] <= 0, 'ingresos_anuales'] = np.nan
X_train.loc[X_train['saldo_cuenta'] == -999, 'saldo_cuenta'] = np.nan

X_test.loc[(X_test['edad'] <= 0) | (X_test['edad'] > 120), 'edad'] = np.nan
X_test.loc[X_test['ingresos_anuales'] <= 0, 'ingresos_anuales'] = np.nan
X_test.loc[X_test['saldo_cuenta'] == -999, 'saldo_cuenta'] = np.nan

print("Nulos detectados en X_train tras sustituir centinelas:")
print(X_train[cols_con_centinelas].isna().sum())

print("\nNulos detectados en X_test tras sustituir centinelas:")
print(X_test[cols_con_centinelas].isna().sum())

# Sanitizamos la columna categórica tipo_cuenta quitando el '?' y normalizando textos
X_train['tipo_cuenta'] = X_train['tipo_cuenta'].replace('?', np.nan)
X_test['tipo_cuenta'] = X_test['tipo_cuenta'].replace('?', np.nan)

X_train['tipo_cuenta'] = X_train['tipo_cuenta'].str.strip().str.capitalize()
X_test['tipo_cuenta'] = X_test['tipo_cuenta'].str.strip().str.capitalize()

Nulos detectados en X_train tras sustituir centinelas:
edad                 9
ingresos_anuales    10
saldo_cuenta         0
dtype: int64

Nulos detectados en X_test tras sustituir centinelas:
edad                2
ingresos_anuales    0
saldo_cuenta        0
dtype: int64


## Imputación de los valores nulos

In [ ]:
# Imputación de variables numéricas y categóricas
# 1. Imputamos por mediana las variables asimétricas
cols_mediana = ['ingresos_anuales', 'antiguedad_meses', 'saldo_cuenta']
imputer_mediana = SimpleImputer(strategy='median')
imputer_mediana.fit(X_train[cols_mediana])

X_train[cols_mediana] = imputer_mediana.transform(X_train[cols_mediana])
X_test[cols_mediana] = imputer_mediana.transform(X_test[cols_mediana])

# 2. Imputamos por media la variable simétrica edad
cols_media = ['edad']
imputer_media = SimpleImputer(strategy='mean')
imputer_media.fit(X_train[cols_media])

X_train[cols_media] = imputer_media.transform(X_train[cols_media])
X_test[cols_media] = imputer_media.transform(X_test[cols_media])

# 3. Imputamos las variables categóricas con la moda más frecuente
cols_categoricas = ['tipo_cuenta', 'canal_preferido', 'nivel_estudios', 'nivel_satisfaccion']
imputer_cat = SimpleImputer(strategy='most_frequent')
imputer_cat.fit(X_train[cols_categoricas])

X_train[cols_categoricas] = imputer_cat.transform(X_train[cols_categoricas])
X_test[cols_categoricas] = imputer_cat.transform(X_test[cols_categoricas])

# Mostramos los parámetros aprendidos en Train
print("Medianas aprendidas por columna:")
for col, val in zip(cols_mediana, imputer_mediana.statistics_):
    print(f" - {col}: {val:.2f}")

print("\nMedia aprendida por columna:")
for col, val in zip(cols_media, imputer_media.statistics_):
    print(f" - {col}: {val:.2f}")

print("\nModas aprendidas en X_train:")
for col, moda in zip(cols_categoricas, imputer_cat.statistics_):
    print(f" - {col}: '{moda}'")

Medianas aprendidas por columna:
 - ingresos_anuales: 27167.64
 - antiguedad_meses: 61.00
 - saldo_cuenta: 5810.57

Media aprendida por columna:
 - edad: 42.09

Modas aprendidas en X_train:
 - tipo_cuenta: 'Corriente'
 - canal_preferido: 'App Movil'
 - nivel_estudios: 'Grado'
 - nivel_satisfaccion: 'Medio'


# Fase 6 - Feature Engineering

## Diseño de un ratio financiero

In [ ]:
# Creación del ratio saldo entre ingresos anuales
X_train['ratio_saldo_ingresos'] = X_train['saldo_cuenta'] / (X_train['ingresos_anuales'] + 1)
X_test['ratio_saldo_ingresos'] = X_test['saldo_cuenta'] / (X_test['ingresos_anuales'] + 1)

In [ ]:
X_train['ratio_saldo_ingresos'].describe()

,ratio_saldo_ingresos
count,480.000000
mean,0.529234
std,0.849952
min,0.000000
25%,0.069123
50%,0.227636
75%,0.634353
max,8.811841


## Diseño de un ratio por producto

In [ ]:
# Creación del saldo medio por cada producto contratado
X_train['saldo_por_producto'] = X_train['saldo_cuenta'] / X_train['num_productos']
X_test['saldo_por_producto'] = X_test['saldo_cuenta'] / X_test['num_productos']

In [ ]:
X_train['saldo_por_producto'].describe()

,saldo_por_producto
count,480.000000
mean,8124.182250
std,13672.264993
min,0.000000
25%,1232.290833
50%,3669.745000
75%,10036.243333
max,146281.940000


## Diseño de una variable agrupada por tramos

In [ ]:
# Agrupamos la edad en tres tramos: Joven, Adulto y Senior
limites_edad = [0, 30, 55, 120]
etiquetas_edad = ['Joven', 'Adulto', 'Senior']

X_train['grupo_edad'] = pd.cut(X_train['edad'], bins=limites_edad, labels=etiquetas_edad)
X_test['grupo_edad'] = pd.cut(X_test['edad'], bins=limites_edad, labels=etiquetas_edad)

In [ ]:
print(X_train['grupo_edad'].value_counts())
print(X_test['grupo_edad'].value_counts())

grupo_edad
Adulto    342
Joven      77
Senior     61
Name: count, dtype: int64
grupo_edad
Adulto    82
Joven     22
Senior    16
Name: count, dtype: int64


## Transformación temporal de escala

In [ ]:
# Convertimos la antigüedad de meses a años
X_train['antiguedad_anios'] = X_train['antiguedad_meses'] / 12.0
X_test['antiguedad_anios'] = X_test['antiguedad_meses'] / 12.0

## Decisión sobre redundancias e integración en el Pipeline

* **Saldo, ingresos y el nuevo ratio:** Me quedo con las tres cosas, el ratio me ayuda a ver si un cliente ahorra poco o mucho comparado con su sueldo, pero el saldo y los ingresos me dicen el dinero real que tiene.
* **Antigüedad en meses o en años:** Me quedo con los años y quito los meses. Prefiero quedarme con los años porque es mejor y  más fácil de entender y quedan en una escala más cómoda sin perder detalle gracias a los decimales.
* **Edad normal o grupos de edad:** Me quedo con la edad y quito los grupos. Si los meto en tres tipos joven, adulto y senior pierdo los años exactos de cada uno, así que prefiero dejar el número tal cual para que el modelo tenga toda la información.

## Investigación: Selección de Características

**a) ¿Qué es la Selección de Características?**
Se trata de quedarme con las columnas que me dan informacion útil y quitar las columnas que no cambian nada, que están repetidas o que solo confunden al modelo.

**b) ¿Qué ventajas le da al modelo?**
- El modelo aprende mucho mejor al no tener en cuenta los datos que no tienen que ver con el abandono.
- El modelo entrena más rápido y gasta menos recursos al tener que trabajar con menos columnas.
- Los resultados son más fáciles de entender y de explicar, porque se ve mejor qué cosas son las que hacen que la gente se vaya.

**c) ¿Cuándo hace falta usarla?**
Cuando tengo una tabla con muchas columnas, cuando el ordenador tarda mucho en procesar los datos o cuando tengo columnas que se relacionan entre ellas.

**d) Cómo funciona SelectKBest:**
Es una prueba que mira cada columna por separado y le pone una nota según lo bien que se relaciona con si el cliente se marcha o se queda. No hace falta entrenar el modelo antes, calcula esa puntuación y se queda con las columnas que tengan las mejores notas y quitando las demás.

# Escalada de Datos

## StandardScaler

In [ ]:
# Escalamos con StandardScaler las columnas simétricas sin valores extremos
cols_standard = ['edad', 'antiguedad_anios']

scaler_std = StandardScaler()
scaler_std.fit(X_train[cols_standard])

print("Parámetros aprendidos por StandardScaler:")
for col, media, desv in zip(cols_standard, scaler_std.mean_, scaler_std.scale_):
    print(f" - {col}: media (μ) = {media:.2f} | desviación (σ) = {desv:.2f}")

# Transformamos Train y Test aprendiendo solo de Train
X_train[cols_standard] = scaler_std.transform(X_train[cols_standard])
X_test[cols_standard] = scaler_std.transform(X_test[cols_standard])

Parámetros aprendidos por StandardScaler:
 - edad: media (μ) = 42.09 | desviación (σ) = 11.59
 - antiguedad_anios: media (μ) = 5.08 | desviación (σ) = 2.72


## RobustScaler

In [ ]:
# Escalamos con RobustScaler las columnas con asimetría y outliers, incluidos los nuevos ratios
cols_robust = ['ingresos_anuales', 'saldo_cuenta', 'ratio_saldo_ingresos', 'saldo_por_producto']

scaler_rob = RobustScaler()
scaler_rob.fit(X_train[cols_robust])

print("Parámetros aprendidos por RobustScaler:")
for col, med, iqr in zip(cols_robust, scaler_rob.center_, scaler_rob.scale_):
    print(f" - {col}: mediana = {med:.2f} | IQR = {iqr:.2f}")

# Transformamos Train y Test aprendiendo solo de Train
X_train[cols_robust] = scaler_rob.transform(X_train[cols_robust])
X_test[cols_robust] = scaler_rob.transform(X_test[cols_robust])

Parámetros aprendidos por RobustScaler:
 - ingresos_anuales: mediana = 27167.64 | IQR = 16136.70
 - saldo_cuenta: mediana = 5810.57 | IQR = 14527.35
 - ratio_saldo_ingresos: mediana = 0.23 | IQR = 0.57
 - saldo_por_producto: mediana = 3669.74 | IQR = 8803.95


# Codificación Ordinal, Escalado y Nominal

In [ ]:
# Codificamos las variables ordinales respetando su orden de menor a mayor
orden_estudios = ['Primaria', 'Secundaria', 'Grado', 'Master', 'Doctorado']
orden_satisfaccion = ['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto']

encoder_ordinal = OrdinalEncoder(categories=[orden_estudios, orden_satisfaccion])

cols_ordinales = ['nivel_estudios', 'nivel_satisfaccion']
encoder_ordinal.fit(X_train[cols_ordinales])

X_train_ord = encoder_ordinal.transform(X_train[cols_ordinales])
X_test_ord = encoder_ordinal.transform(X_test[cols_ordinales])

In [ ]:
# Escalamos las variables ordinales entre 0 y 1 con MinMaxScaler
scaler_ord = MinMaxScaler()
X_train_ord_scaled = scaler_ord.fit_transform(X_train_ord)
X_test_ord_scaled = scaler_ord.transform(X_test_ord)

In [ ]:
# Codificamos las variables nominales en columnas dummy con OneHotEncoder
cols_nominales = ['tipo_cuenta', 'canal_preferido']

encoder_ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

encoder_ohe.fit(X_train[cols_nominales])

X_train_ohe = encoder_ohe.transform(X_train[cols_nominales])
X_test_ohe = encoder_ohe.transform(X_test[cols_nominales])

nombres_dummies = encoder_ohe.get_feature_names_out(cols_nominales)
print(f"Número de características dummy generadas: {len(nombres_dummies)}")
print("Nuevas columnas generadas:")
for nombre in nombres_dummies:
    print(f" - {nombre}")

Número de características dummy generadas: 9
Nuevas columnas generadas:
 - tipo_cuenta_Ahorro
 - tipo_cuenta_Corriente
 - tipo_cuenta_Inversion
 - tipo_cuenta_Nomina
 - canal_preferido_App Movil
 - canal_preferido_Cajero Antiguo
 - canal_preferido_Oficina
 - canal_preferido_Telefono
 - canal_preferido_Web


# Ensamble Manual y Auditoría Dimensional

In [ ]:
# Unimos las ramas numérica, ordinal escalada y nominal con np.hstack
# Incluimos las columnas cuantitativas y los nuevos ratios seleccionados
cols_numericas = [
    'edad',
    'ingresos_anuales',
    'antiguedad_anios',
    'saldo_cuenta',
    'num_productos',
    'tiene_tarjeta',
    'ratio_saldo_ingresos',
    'saldo_por_producto'
]

nombres_columnas_final = (
    cols_numericas +
    cols_ordinales +
    list(encoder_ohe.get_feature_names_out(cols_nominales))
)

X_train_final = pd.DataFrame(
    np.hstack([
        X_train[cols_numericas].to_numpy(),
        X_train_ord_scaled,
        X_train_ohe
    ]),
    columns=nombres_columnas_final,
    index=X_train.index
)

X_test_final = pd.DataFrame(
    np.hstack([
        X_test[cols_numericas].to_numpy(),
        X_test_ord_scaled,
        X_test_ohe
    ]),
    columns=nombres_columnas_final,
    index=X_test.index
)

print(f"Dimensiones de X_train_final: {X_train_final.shape}")
print(f"Dimensiones de X_test_final:  {X_test_final.shape}")

# Mostramos las primeras 5 filas del resultado
display(X_train_final.head(5))
display(X_test_final.head(5))

Dimensiones de X_train_final: (480, 19)
Dimensiones de X_test_final:  (120, 19)


,edad,ingresos_anuales,antiguedad_anios,saldo_cuenta,num_productos,tiene_tarjeta,ratio_saldo_ingresos,saldo_por_producto,nivel_estudios,nivel_satisfaccion,tipo_cuenta_Ahorro,tipo_cuenta_Corriente,tipo_cuenta_Inversion,tipo_cuenta_Nomina,canal_preferido_App Movil,canal_preferido_Cajero Antiguo,canal_preferido_Oficina,canal_preferido_Telefono,canal_preferido_Web
145,0.251078,-0.589291,-1.012232,-0.292502,2.0,1.0,-0.246316,-0.328159,0.25,0.50,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
9,0.078565,1.730610,0.000064,0.313117,2.0,0.0,-0.070077,0.171504,0.50,0.50,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
375,0.251078,-0.244450,0.981684,-0.053864,2.0,0.0,-0.019697,-0.131272,0.50,0.75,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
523,-1.474055,-0.773509,-0.858854,1.396014,3.0,1.0,2.740226,0.571020,0.25,0.00,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
188,0.337335,-0.416831,-1.257637,3.330132,3.0,1.0,4.287037,1.634846,1.00,1.00,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


,edad,ingresos_anuales,antiguedad_anios,saldo_cuenta,num_productos,tiene_tarjeta,ratio_saldo_ingresos,saldo_por_producto,nivel_estudios,nivel_satisfaccion,tipo_cuenta_Ahorro,tipo_cuenta_Corriente,tipo_cuenta_Inversion,tipo_cuenta_Nomina,canal_preferido_App Movil,canal_preferido_Cajero Antiguo,canal_preferido_Oficina,canal_preferido_Telefono,canal_preferido_Web
110,0.509848,-0.104868,-1.257637,1.369122,1.0,1.0,1.382005,2.502345,0.50,0.50,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
419,-0.352718,-0.147841,-1.257637,3.381987,2.0,0.0,3.519423,2.703466,0.75,0.25,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
565,-1.646569,0.585563,-0.766827,0.074383,1.0,1.0,-0.069785,0.365904,0.50,0.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
77,-0.611488,-0.411595,0.306820,0.889287,2.0,0.0,1.211550,0.646871,0.50,0.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
181,-1.560312,1.147978,1.809926,0.440748,2.0,1.0,0.070160,0.276806,0.00,0.25,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
